In [ ]:
import pandas as pd
import pandera as pa
from pandera.typing import DataFrame, Series
from datetime import datetime
import pyarrow as pa_arrow

In [ ]:
df = pd.read_csv('dataset1.csv')
print(df.shape)

In [ ]:
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(df.shape)

In [ ]:
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())

In [ ]:
class MusicDataSchema(pa.DataFrameModel):
    track_id: Series[str] = pa.Field(nullable=False, str_length=22)
    artists: Series[str] = pa.Field(nullable=False, str_length=(2, 512))
    album_name: Series[str] = pa.Field(nullable=False, str_length=(2, 512))
    track_name: Series[str] = pa.Field(nullable=False, str_length=(2, 512))
    popularity: Series[int] = pa.Field(ge=0, le=100, nullable=False)
    duration_ms: Series[int] = pa.Field(gt=0, le=5237760, nullable=False)
    explicit: Series[bool] = pa.Field(nullable=False)
    danceability: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    energy: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    key: Series[int] = pa.Field(ge=0, le=11, nullable=False)
    loudness: Series[float] = pa.Field(ge=-45, le=5, nullable=False)
    mode: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    speechiness: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    acousticness: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    instrumentalness: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    liveness: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    valence: Series[float] = pa.Field(ge=0, le=1, nullable=False)
    tempo: Series[float] = pa.Field(ge=0, le=256, nullable=False)
    time_signature: Series[int] = pa.Field(ge=0, le=5, nullable=False)
    track_genre: Series[str] = pa.Field(nullable=False, isin=UNIQUE_GENRES)
    
    class Config:
        strict = True
        coerce = True

In [ ]:
try:
    validated_df = MusicDataSchema.validate(df, lazy=True)
    errors_df = pd.DataFrame(columns=['row', 'column', 'error_type', 'value', 'message'])
except pa.errors.SchemaErrors as e:
    failure_cases = e.failure_cases
    errors_list = []
    for _, row in failure_cases.iterrows():
        errors_list.append({
            'row': int(row['index']) if pd.notna(row.get('index')) else None,
            'column': row.get('column', 'N/A'),
            'error_type': row.get('check', 'schema_error'),
            'value': str(row.get('failure_case', 'N/A')),
            'message': f"{row.get('check', 'Error')} failed for value: {row.get('failure_case', 'N/A')}"
        })
    errors_df = pd.DataFrame(errors_list) 
    mask = ~errors_df['value'].str.contains('ValueError', na=False)
    errors_df = errors_df[mask].reset_index(drop=True)
    print(len(errors_df))

In [ ]:
# Проверка на 114 уникальных жанров
n_genres = df['track_genre'].nunique()
if n_genres != 114:
    errors_list.append({
        'row': 'all',
        'column': 'track_genre',
        'error_type': 'unique_count',
        'value': n_genres,
        'message': f'Должно быть 114 уникальных жанров, найдено: {n_genres}'
    })
    errors_df = pd.DataFrame(errors_list)

In [ ]:
report_filename = f'SharafutdinovNil_Task3_report.parquet'
errors_df.to_parquet(report_filename, engine='pyarrow')